Se importaron los archivos de .csv

In [2]:
import pandas as pd
import numpy as np

In [3]:
aisles = pd.read_csv("../data/raw/aisles.csv", sep=";")

departments = pd.read_csv("../data/raw/departments.csv", sep=";")

orders = pd.read_csv("../data/raw/instacart_orders.csv", sep=";")

order_prods = pd.read_csv("../data/raw/order_products.csv", sep=";")

products = pd.read_csv("../data/raw/products.csv", sep=";")

LECCION 2:LIMPIEZA Y ESTANDARIZACION

In [31]:
print(orders.columns.tolist())
print("Índice:", orders.index.name)

['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Índice: None


TIPO DE DATO CON asktype

In [30]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 26.8 MB


Después: 11.0 MB


NULOS :COMPROBAR LA CAUSA ANTES DE DESIDIR

In [29]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


In [28]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

Series([], Name: count, dtype: int64)
0


TEXTO CON LOS METODOS .STR

In [27]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 0
Espacio invisible: 0
0 0


DUPLICADOS IMPLICITOS

In [6]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


LIMPIEZA REUTILIZABLE CON FUNCIONES

In [7]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


VERIFICACION DE LA LIMPIEZA

In [19]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

LABORATORIO GUIADO 2:PIPELINE DE LIMPIEZA CON INFORME DE CAMBIO

PASO 1 :RECARGAR LOS DATOS CRUDOS

In [21]:
orders_crudo = pd.read_csv("../data/raw/instacart_orders.csv", sep=";")

products_crudo = pd.read_csv("../data/raw/products.csv", sep=";")

op_crudo = pd.read_csv("../data/raw/order_products.csv", sep=";")

print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


PASO 2:UNA FUNCION QUE ANOTA EL ESTADO DE LA TABLA

In [22]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

PASO 3:UN PASO DE LIMPIEZA POR FUNCION

In [23]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

PASO 4:ENCADENAR LOS PASOS CON PIPE

In [24]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

PASO 5:INFORME DE CAMBIOS

In [25]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,5.1
4,products,1 nombres,49694,0,5.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


PASO 6:VALIDAR LOS RESULTADOS

In [32]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
False True


RETO 2

1. Comprueba si order_prods tiene filas duplicadas exactas, si se repite algún par order_id + product_id y si reordered solo contiene 0 y 1.

In [4]:

print("Duplicados exactos:", order_prods.duplicated().sum())


print(
    "Combinaciones repetidas:",
    order_prods.duplicated(subset=["order_id", "product_id"]).sum()
)


print("Valores encontrados:", order_prods["reordered"].unique())


print("¿Solo contiene 0 y 1?:", order_prods["reordered"].isin([0, 1]).all())

Duplicados exactos: 0
Combinaciones repetidas: 0
Valores encontrados: [0 1]
¿Solo contiene 0 y 1?: True


2. Comprueba si aisles y departments tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?

In [5]:

print("Aisles duplicados:", aisles["aisle"].duplicated().sum())
print("Departments duplicados:", departments["department"].duplicated().sum())


print(
    "Aisles con espacios:",
    (aisles["aisle"] != aisles["aisle"].str.strip()).sum()
)

print(
    "Departments con espacios:",
    (departments["department"] != departments["department"].str.strip()).sum()
)


print(aisles["aisle"].unique())
print(departments["department"].unique())

Aisles duplicados: 0
Departments duplicados: 0
Aisles con espacios: 0
Departments con espacios: 0
<StringArray>
[     'prepared soups salads',          'specialty cheeses',
        'energy granola bars',              'instant foods',
 'marinades meat preparation',                      'other',
              'packaged meat',            'bakery desserts',
                'pasta sauce',           'kitchen supplies',
 ...
        'trail mix snack mix',              'feminine care',
          'body lotions soap',       'tortillas flat bread',
    'frozen appetizers sides',   'hot cereal pancake mixes',
                  'dry pasta',                     'beauty',
 'muscles joints pain relief', 'specialty wines champagnes']
Length: 134, dtype: str
<StringArray>
[         'frozen',           'other',          'bakery',         'produce',
         'alcohol',   'international',       'beverages',            'pets',
 'dry goods pasta',            'bulk',   'personal care',    'meat seafood',
    

3. Investiga los 836 nulos de add_to_cart_order: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [6]:

nulos = order_prods[order_prods["add_to_cart_order"].isna()]

print("Total de valores nulos:", len(nulos))


pedidos_afectados = nulos["order_id"].unique()

print("Cantidad de pedidos afectados:", len(pedidos_afectados))


productos_por_pedido = (
    order_prods.groupby("order_id")
    .size()
)

cantidades = productos_por_pedido.loc[pedidos_afectados]

print("Cantidad mínima de productos:", cantidades.min())
print("Cantidad máxima de productos:", cantidades.max())
print("Resumen de productos por pedido:")
print(cantidades.describe())

Total de valores nulos: 836
Cantidad de pedidos afectados: 70
Cantidad mínima de productos: 65
Cantidad máxima de productos: 127
Resumen de productos por pedido:
count     70.000000
mean      75.942857
std       12.898585
min       65.000000
25%       67.000000
50%       71.000000
75%       78.000000
max      127.000000
dtype: float64


4. Carga de nuevo products.csv en una variable products_crudo y cuenta cuántos nombres (no nulos) cambiaron con la estandarización.

In [7]:

products_crudo = pd.read_csv("../data/raw/products.csv", sep=";")


comparacion = products_crudo[
    ["product_id", "product_name"]
].merge(
    products[["product_id", "product_name"]],
    on="product_id",
    suffixes=("_original", "_limpio")
)


cambiaron = (
    comparacion["product_name_original"].notna()
    & comparacion["product_name_limpio"].notna()
    & (
        comparacion["product_name_original"]
        != comparacion["product_name_limpio"]
    )
)

print("Nombres que cambiaron:", cambiaron.sum())

Nombres que cambiaron: 0


5. Crea una Series con el nombre del día de cada pedido y conviértela a una categoría ordenada de domingo a sábado. Compara la memoria del texto con la de la categoría.

In [8]:

dias = {
    0: "Domingo",
    1: "Lunes",
    2: "Martes",
    3: "Miércoles",
    4: "Jueves",
    5: "Viernes",
    6: "Sábado"
}

dias_semana = orders["order_dow"].map(dias)


orden_dias = [
    "Domingo", "Lunes", "Martes",
    "Miércoles", "Jueves", "Viernes", "Sábado"
]

dias_categoria = pd.Categorical(
    dias_semana,
    categories=orden_dias,
    ordered=True
)


memoria_texto = dias_semana.memory_usage(deep=True)
memoria_categoria = dias_categoria.memory_usage(deep=True)

print("Memoria como texto:", memoria_texto, "bytes")
print("Memoria como categoría:", memoria_categoria, "bytes")

print("¿La categoría ahorra memoria?:", memoria_categoria < memoria_texto)

Memoria como texto: 28568422 bytes
Memoria como categoría: 479688 bytes
¿La categoría ahorra memoria?: True


6. ¿Cuántas filas de order_prods corresponden a productos 'Unknown' y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica.

In [9]:

unknown = products[products["product_name"] == "Unknown"]


ids_unknown = unknown["product_id"]


registros_unknown = order_prods[
    order_prods["product_id"].isin(ids_unknown)
]


print("Registros de productos Unknown:", len(registros_unknown))

print(
    "Pedidos afectados:",
    registros_unknown["order_id"].nunique()
)

Registros de productos Unknown: 0
Pedidos afectados: 0


7. En una copia de departments, reemplaza 'missing' por un nulo real (pd.NA). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.

In [10]:

departments_copia = departments.copy()


departments_copia["department"] = (
    departments_copia["department"].replace("missing", pd.NA)
)


print("Valores originales:")
print(departments["department"].unique())

print("\nValores después del reemplazo:")
print(departments_copia["department"].unique())

print(
    "\nCantidad de valores nulos:",
    departments_copia["department"].isna().sum()
)

Valores originales:
<StringArray>
[         'frozen',           'other',          'bakery',         'produce',
         'alcohol',   'international',       'beverages',            'pets',
 'dry goods pasta',            'bulk',   'personal care',    'meat seafood',
          'pantry',       'breakfast',    'canned goods',      'dairy eggs',
       'household',          'babies',          'snacks',            'deli',
         'missing']
Length: 21, dtype: str

Valores después del reemplazo:
<StringArray>
[         'frozen',           'other',          'bakery',         'produce',
         'alcohol',   'international',       'beverages',            'pets',
 'dry goods pasta',            'bulk',   'personal care',    'meat seafood',
          'pantry',       'breakfast',    'canned goods',      'dairy eggs',
       'household',          'babies',          'snacks',            'deli',
               nan]
Length: 21, dtype: str

Cantidad de valores nulos: 1
